In [1]:
import numpy as np
import sympy as sp
from scipy.optimize import linprog

In [2]:
G = [
    [30, 95, 65, 105, 20],
    [20, 75, 35, 70, 50]
]

## Застосування принципу домінування

In [3]:
def clear_dominant_rows(G):
    for i in range(len(G)):
        for j in range(i + 1, len(G)):
            if all(G[i][k] <= G[j][k] for k in range(len(G[i]))):
                G[i] = [0] * len(G[i])
                break
            elif all(G[j][k] <= G[i][k] for k in range(len(G[j]))):
                G[j] = [0] * len(G[j])
                break
    return G


def clear_dominant_columns(G):
    for i in range(len(G[0])):
        for j in range(i + 1, len(G[0])):
            if all(G[k][i] <= G[k][j] for k in range(len(G))):
                for k in range(len(G)):
                    G[k][i] = 0
                break
            elif all(G[k][j] <= G[k][i] for k in range(len(G))):
                for k in range(len(G)):
                    G[k][j] = 0
                break
    return G


def remove_zeros(matrix):
    non_zero_rows = ~np.all(matrix == 0, axis=1)
    filtered_matrix = matrix[non_zero_rows]

    non_zero_columns = ~np.all(filtered_matrix == 0, axis=0)
    filtered_matrix = filtered_matrix[:, non_zero_columns]

    return filtered_matrix


def dominant(G):
    G = np.array(G)
    G = remove_zeros(G)
    G = clear_dominant_rows(G)
    G = remove_zeros(G)
    G = clear_dominant_columns(G)
    G = remove_zeros(G)
    return G


In [4]:
G = dominant(G)
G

array([[ 95, 105],
       [ 75,  70]])

## Сідлова точка

In [5]:
sidl_i = np.argmax(np.min(G, axis=1))
sidl_j = np.argmin(np.max(G, axis=0))
print(f'Сідлова точка: {G[sidl_i][sidl_j]}, Стратегія А: {sidl_i + 1}, Стратегія B: {sidl_j + 1}')

Сідлова точка: 95, Стратегія А: 1, Стратегія B: 1


## Розв’язати геометричну задачу для гравця 𝐵.

In [6]:
q = sp.symbols('q')
H1 = G[0][0] * q + G[0][1] * (1 - q)
H2 = G[1][0] * q + G[1][1] * (1 - q)

eq = sp.Eq(H1, H2)
sol = sp.solve(eq, q)
print(f'точка перетину q = {sol[0]}')

точка перетину q = 7/3


Оскільки 7/3 > 1, то оптимальна стратегія q = 0

In [7]:
v = np.max(G[..., 0])
print(f'Ціна гри для гравця B: {v}')

Ціна гри для гравця B: 95


## Задача ЛП

### Для гравця А

In [8]:
c = np.array([0, 0, -1])

# Обмеження: G^T @ x >= v → -G^T @ x <= -v
A_ub = np.array([
    [-95, -75, 1],
    [-105, -70, 1]
])
b_ub = [0, 0]

# Додаткове обмеження: x1 + x2 = 1
A_eq = [[1, 1, 0]]
b_eq = [1]

bounds = [(0, 1), (0, 1), (None, None)]

res_A = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method="highs")
print("Результат для гравця A:")
print(f"  Стратегії: x1 = {res_A.x[0]:.3f}, x2 = {res_A.x[1]:.3f}")
print(f"  Ціна гри: v = {res_A.x[2]:.2f}")

Результат для гравця A:
  Стратегії: x1 = 1.000, x2 = 0.000
  Ціна гри: v = 95.00


### Для гравця B

In [9]:
c = -c
A_ub_B = np.array([
    [95, 105, -1],
    [75, 70, -1]
])
b_ub_B = [0, 0]

# y1 + y2 = 1
A_eq_B = [[1, 1, 0]]
b_eq_B = [1]

bounds_B = [(0, 1), (0, 1), (None, None)]

res_B = linprog(c, A_ub=A_ub_B, b_ub=b_ub_B, A_eq=A_eq_B, b_eq=b_eq_B, bounds=bounds_B, method="highs")

print("Результат для гравця B:")
print(f"  Стратегії: y1 = {res_B.x[0]:.3f}, y2 = {res_B.x[1]:.3f}")
print(f"  Ціна гри: v = {res_B.x[2]:.2f}")

Результат для гравця B:
  Стратегії: y1 = 1.000, y2 = 0.000
  Ціна гри: v = 95.00
